# Modelo XGBoost — Fase 7: evaluación final en test

Este notebook evalúa en test el modelo XGBoost final, que ya está congelado en `models/xgboost/`. Sus piezas son:

- hiperparámetros elegidos en `03_tuning.ipynb` (`learning_rate=0.3`, `max_depth=4`, `min_child_weight=1`);
- `scale_pos_weight` = benignos / ataques de `y_train` (variante C de `02b_ablacion_smote.ipynb`);
- umbral elegido en `04_umbral.ipynb` (`UMBRAL_FINAL = 0.87`, regla de mayor F2 con recall >= 0.75);
- modelo guardado y verificado en `05_verificacion_modulo.ipynb`.

El modelo se carga con `load_xgboost_model()` de `src/models/xgboost/model.py`. **No se reentrena.** Todo corresponde al dataset corregido (443,614 filas).

**Con test no se ajusta ni se re-elige nada:** ni el umbral, ni los hiperparámetros, ni las features. El modelo se aplica a test una sola vez, con el umbral tal como quedó cerrado en val.

**Comparación con la GRU sobre las mismas filas.** La GRU se evaluó sobre un subconjunto de test, cuya lista está en `results/gru/03_evaluacion_test/idx_test_comun.npy`: 66,454 etiquetas del índice de filas que devuelve `train_test_split_by_flow` (no posiciones). Además de las métricas sobre test completo, se reportan las mismas métricas restringidas a esas filas. El modelo predice test completo una sola vez; las métricas del subconjunto común salen de filtrar las probabilidades ya guardadas, sin una segunda predicción.

Secciones:

1. Carga del modelo y sanidad previa a test: metadatos contra el módulo (incluido `scale_pos_weight` y sus conteos de `y_train`), verificación en val (FP 11 y FN 3 a umbral 0.87) y una tabla informativa de val con otros umbrales. Si algo no coincide, el notebook se detiene antes de tocar test.
2. Test (no ejecutar más de una vez): carga y verificaciones de test, verificación de la lista de filas comunes con la GRU, predicción única con guarda y probabilidades guardadas en disco.
3. Métricas en test completo y en el subconjunto común, con el mismo procedimiento y a partir de las probabilidades guardadas: umbral principal, sensibilidad a otros umbrales, recall por tipo de ataque e intervalos de confianza por bootstrap.
4. Resumen.

In [1]:
import json
import sys
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
# Así el notebook funciona sin importar desde dónde se abra el kernel.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, confusion_matrix

from src.evaluation.metrics import evaluate_model
from src.models.xgboost.model import (
    HIPERPARAMETROS,
    MODEL_DIR,
    UMBRAL_FINAL,
    apply_threshold,
    calcular_scale_pos_weight,
    load_xgboost_model,
    predict_xgboost_proba,
)
from src.preprocessing.clean import (
    COLUMNA_ETIQUETA,
    ETIQUETA_BENIGNA,
    ETIQUETAS_LOW_AND_SLOW,
    cargar_dataset_limpio,
    train_test_split_by_flow,
)

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
print(f"Python:        {sys.executable}")
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"Datos:         {DATA_DIR}")
print(f"Modelo:        {MODEL_DIR}")
print(f"xgboost {xgb.__version__}")

Python:        C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\venv\Scripts\python.exe
Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
Datos:         C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\data\processed\xgboost
Modelo:        C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\models\xgboost
xgboost 3.3.0


## 1. Carga del modelo y sanidad previa a test

### Metadatos del modelo congelado

Los metadatos guardados junto al modelo tienen que coincidir con el módulo:

- hiperparámetros iguales a `HIPERPARAMETROS`;
- umbral igual a `UMBRAL_FINAL` (0.87);
- best_iteration 68, que es también el que tiene que traer el modelo cargado.

`scale_pos_weight` no es una constante, así que se verifica contra `y_train`. Se cuentan benignos y ataques en `y_train.csv` (solo etiquetas de train) y tienen que dar 302,299 y 8,230. Los metadatos tienen que traer esos mismos conteos (`n_benignos_train`, `n_ataques_train`) y `scale_pos_weight` igual a su cociente. El booster cargado tiene que tener ese mismo valor; lo guarda en float32, así que ahí se compara en float32. Si algo no coincide, el notebook se detiene.

In [2]:
modelo, metadatos = load_xgboost_model()
print(f"Cargado de {MODEL_DIR.relative_to(RAIZ_REPO)}:")
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

UMBRAL_ESPERADO = 0.87
BEST_ITERATION_ESPERADO = 68
N_BENIGNOS_TRAIN_ESPERADO = 302_299
N_ATAQUES_TRAIN_ESPERADO = 8_230

# scale_pos_weight se recalcula de y_train (solo etiquetas de train) para compararlo con los metadatos.
PESOS = calcular_scale_pos_weight(pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy())
spw_booster = json.loads(modelo.get_booster().save_config())["learner"]["objective"]["reg_loss_param"]["scale_pos_weight"]

problemas = []
if UMBRAL_FINAL != UMBRAL_ESPERADO:
    problemas.append(f"UMBRAL_FINAL del módulo es {UMBRAL_FINAL}, se esperaba {UMBRAL_ESPERADO}")
if metadatos["umbral"] != UMBRAL_FINAL:
    problemas.append(f"umbral de los metadatos {metadatos['umbral']} != UMBRAL_FINAL {UMBRAL_FINAL}")
if metadatos["hiperparametros"] != HIPERPARAMETROS:
    problemas.append(f"hiperparámetros de los metadatos {metadatos['hiperparametros']} != HIPERPARAMETROS {HIPERPARAMETROS}")
if metadatos["best_iteration"] != BEST_ITERATION_ESPERADO:
    problemas.append(f"best_iteration de los metadatos {metadatos['best_iteration']} != {BEST_ITERATION_ESPERADO}")
if modelo.best_iteration != BEST_ITERATION_ESPERADO:
    problemas.append(f"best_iteration del modelo cargado {modelo.best_iteration} != {BEST_ITERATION_ESPERADO}")
if (PESOS["n_benignos_train"], PESOS["n_ataques_train"]) != (N_BENIGNOS_TRAIN_ESPERADO, N_ATAQUES_TRAIN_ESPERADO):
    problemas.append(f"y_train.csv tiene {PESOS['n_benignos_train']:,} benignos y {PESOS['n_ataques_train']:,} ataques, "
                     f"se esperaban {N_BENIGNOS_TRAIN_ESPERADO:,} y {N_ATAQUES_TRAIN_ESPERADO:,}")
for clave in ("scale_pos_weight", "n_benignos_train", "n_ataques_train"):
    if metadatos.get(clave) != PESOS[clave]:
        problemas.append(f"{clave} de los metadatos {metadatos.get(clave)} != {PESOS[clave]} (calculado de y_train)")
if np.float32(float(spw_booster)) != np.float32(PESOS["scale_pos_weight"]):
    problemas.append(f"scale_pos_weight del booster cargado {spw_booster} != {PESOS['scale_pos_weight']} en float32")

if problemas:
    raise RuntimeError("Los metadatos no coinciden con el modelo congelado: " + "; ".join(problemas))
print(f"\nscale_pos_weight = {PESOS['n_benignos_train']:,} / {PESOS['n_ataques_train']:,} = {PESOS['scale_pos_weight']} "
      f"(metadatos {metadatos['scale_pos_weight']}, booster {spw_booster})")
print(f"Metadatos OK: coinciden con HIPERPARAMETROS, UMBRAL_FINAL {UMBRAL_FINAL:g}, "
      f"best_iteration {BEST_ITERATION_ESPERADO} y scale_pos_weight de y_train.")

Cargado de models\xgboost:
{
  "umbral": 0.87,
  "hiperparametros": {
    "learning_rate": 0.3,
    "max_depth": 4,
    "min_child_weight": 1,
    "n_estimators": 1000,
    "early_stopping_rounds": 50,
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
  },
  "scale_pos_weight": 36.73134872417983,
  "n_benignos_train": 302299,
  "n_ataques_train": 8230,
  "best_iteration": 68
}



scale_pos_weight = 302,299 / 8,230 = 36.73134872417983 (metadatos 36.73134872417983, booster 36.7313499)
Metadatos OK: coinciden con HIPERPARAMETROS, UMBRAL_FINAL 0.87, best_iteration 68 y scale_pos_weight de y_train.


### Verificación en val

Con `X_val` / `y_val` el modelo cargado tiene que dar FP 11 y FN 3 a umbral 0.87, igual que en `04_umbral.ipynb` y `05_verificacion_modulo.ipynb`. Si no, el notebook se detiene acá, antes de tocar test.

Val se carga con los mismos asserts que en 05: 66,541 filas, 1,884 ataques y 55 features, sin `Destination Port`, `Source Port`, `Protocol`, `Fwd PSH Flags`, `Timestamp` ni `bloque`. Las columnas de val, con su orden, se guardan para compararlas con las de test.

In [3]:
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()

N_VAL_ESPERADO = 66_541
N_ATAQUES_VAL_ESPERADO = 1_884
assert len(X_val) == N_VAL_ESPERADO, f"Se esperaban {N_VAL_ESPERADO:,} filas en val, hay {len(X_val):,}"
assert y_val.sum() == N_ATAQUES_VAL_ESPERADO, f"Se esperaban {N_ATAQUES_VAL_ESPERADO:,} ataques en val, hay {y_val.sum():,}"

N_FEATURES_ESPERADO = 55
assert X_val.shape[1] == N_FEATURES_ESPERADO, f"Se esperaban {N_FEATURES_ESPERADO} features, hay {X_val.shape[1]}"

COLUMNAS_EXCLUIDAS = ["Destination Port", "Source Port", "Protocol", "Fwd PSH Flags", "Timestamp", "bloque"]
presentes = [c for c in COLUMNAS_EXCLUIDAS if c in X_val.columns]
assert not presentes, f"Columnas que no deberían estar en X_val: {presentes}"

COLUMNAS_VAL = list(X_val.columns)
print(f"val: X {X_val.shape} | y {y_val.shape} | ataques: {y_val.sum()} ({y_val.mean():.4%})")

proba_val = predict_xgboost_proba(modelo, X_val)
tn_val, fp_val, fn_val, tp_val = confusion_matrix(y_val, apply_threshold(proba_val), labels=[0, 1]).ravel()

REFERENCIA_VAL = {"FP": 11, "FN": 3}
obtenido_val = {"FP": int(fp_val), "FN": int(fn_val)}
print(f"\nUmbral {UMBRAL_FINAL:g} en val -> TN {tn_val} | FP {fp_val} | FN {fn_val} | TP {tp_val}")
print(f"Esperado (04/05): {REFERENCIA_VAL}")
print(f"Obtenido:         {obtenido_val}")

if obtenido_val != REFERENCIA_VAL:
    raise RuntimeError(
        "El modelo cargado NO reproduce val (FP 11, FN 3 a umbral 0.87). "
        "Se detiene el notebook antes de tocar test."
    )
print("\nSanidad previa a test OK: el modelo cargado reproduce val.")

val: X (66541, 55) | y (66541,) | ataques: 1884 (2.8313%)

Umbral 0.87 en val -> TN 64646 | FP 11 | FN 3 | TP 1881
Esperado (04/05): {'FP': 11, 'FN': 3}
Obtenido:         {'FP': 11, 'FN': 3}

Sanidad previa a test OK: el modelo cargado reproduce val.


### Tabla informativa en val (no elige ni cambia nada)

Solo como información, se muestran las métricas en val con otros umbrales además del 0.87 elegido. 0.20 es la otra moda del bootstrap de `04_umbral.ipynb`, y 0.80, 0.90 y 0.95 rodean al umbral elegido. El umbral del modelo sigue siendo `UMBRAL_FINAL`. Val ya se usó para el early stopping, para los hiperparámetros y para el umbral, así que estas métricas son optimistas.

`metricas_umbral` se define acá y se reutiliza en la sección 3 para la sensibilidad en test. F1 se calcula como `2·TP / (2·TP + FP + FN)` y F2 como `5·TP / (5·TP + 4·FN + FP)`, igual que en `04_umbral.ipynb`.

In [4]:
def metricas_umbral(y: np.ndarray, proba: np.ndarray, umbral: float) -> dict:
    """Precision, recall, F1, F2 y matriz de confusión para un umbral, con apply_threshold del módulo."""
    tn, fp, fn, tp = confusion_matrix(y, apply_threshold(proba, umbral), labels=[0, 1]).ravel()
    return {
        "umbral": umbral,
        "precision": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
        "recall": float(tp / (tp + fn)),
        "f1": float(2 * tp / (2 * tp + fp + fn)),
        "f2": float(5 * tp / (5 * tp + 4 * fn + fp)),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }


UMBRALES_INFORMATIVOS_VAL = [0.5, 0.20, 0.80, 0.87, 0.90, 0.95]
tabla_val_informativa = pd.DataFrame([metricas_umbral(y_val, proba_val, u) for u in UMBRALES_INFORMATIVOS_VAL])
tabla_val_informativa.insert(
    1, "rol", ["elegido (UMBRAL_FINAL)" if u == UMBRAL_FINAL else "informativo" for u in UMBRALES_INFORMATIVOS_VAL]
)
print("Val — tabla INFORMATIVA (no elige ni cambia nada):")
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(tabla_val_informativa[["umbral", "rol", "precision", "recall", "f1", "f2", "FP", "FN"]].to_string(index=False))

Val — tabla INFORMATIVA (no elige ni cambia nada):


 umbral                    rol  precision  recall     f1     f2  FP  FN
 0.5000            informativo     0.9931  0.9984 0.9958 0.9973  13   3
 0.2000            informativo     0.9921  0.9989 0.9955 0.9976  15   2
 0.8000            informativo     0.9937  0.9984 0.9960 0.9975  12   3
 0.8700 elegido (UMBRAL_FINAL)     0.9942  0.9984 0.9963 0.9976  11   3
 0.9000            informativo     0.9942  0.9958 0.9950 0.9954  11   8
 0.9500            informativo     0.9940  0.9687 0.9812 0.9736  11  59


## 2. Test — NO EJECUTAR MÁS DE UNA VEZ

Esta sección usa test. **El modelo se aplica a test una sola vez**, en la celda de predicción, y las probabilidades se guardan en `reports/xgboost/probabilidades_test.csv`. Esa celda tiene una guarda: si el archivo ya existe, no vuelve a predecir y avisa que test ya fue evaluado. El resto de las celdas no usa el modelo:

- la carga de test, las verificaciones de test y la verificación de la lista de filas comunes con la GRU son solo sobre los datos;
- las métricas, el recall por tipo y los intervalos leen las probabilidades del archivo.

Por eso se pueden volver a ejecutar sin volver a predecir.

Todas las verificaciones sobre los datos de test se hacen antes de predecir: tamaño, ataques, columnas, reconstrucción de las etiquetas por tipo y lista de filas comunes. Así, si alguna falla, se puede corregir sin haber usado la predicción.

`*.csv` está en `.gitignore`, así que el archivo de probabilidades (y con él la guarda) queda solo en esta copia local del repo. Para versionarlo hace falta `git add -f`.

In [5]:
REPORTES_XGB_DIR = RAIZ_REPO / "reports" / "xgboost"
METRICAS_DIR = RAIZ_REPO / "reports" / "metrics"
RUTA_X_TEST = DATA_DIR / "X_test.csv"
RUTA_Y_TEST = DATA_DIR / "y_test.csv"
RUTA_PROBA_TEST = REPORTES_XGB_DIR / "probabilidades_test.csv"
RUTA_RESULTADOS_TEST = REPORTES_XGB_DIR / "resultados_test.json"
RUTA_DATASET_LIMPIO = RAIZ_REPO / "data" / "processed" / "wednesday_low_and_slow_clean.csv"
# Filas de test que evaluó la GRU: etiquetas del índice que devuelve train_test_split_by_flow, no posiciones.
RUTA_IDX_COMUN = RAIZ_REPO / "results" / "gru" / "03_evaluacion_test" / "idx_test_comun.npy"
# evaluate_model escribe METRICAS_DIR / f"{model_name.lower()}.json": uno para test completo y otro, aparte,
# para el subconjunto común con la GRU.
NOMBRE_METRICAS = "XGBoost"
NOMBRE_METRICAS_COMUN = "XGBoost_comun"
RUTA_METRICAS = METRICAS_DIR / f"{NOMBRE_METRICAS.lower()}.json"
RUTA_METRICAS_COMUN = METRICAS_DIR / f"{NOMBRE_METRICAS_COMUN.lower()}.json"

N_DATASET_LIMPIO_ESPERADO = 443_614
N_TEST_ESPERADO = 66_544
N_ATAQUES_TEST_ESPERADO = 1_175
N_POR_TIPO_ESPERADO = {"DoS slowloris": 915, "DoS Slowhttptest": 260}
N_COMUN_ESPERADO = 66_454
assert set(N_POR_TIPO_ESPERADO) == set(ETIQUETAS_LOW_AND_SLOW), "Los tipos esperados no coinciden con clean.py"

estado = "YA EXISTE: test ya fue evaluado" if RUTA_PROBA_TEST.exists() else "todavía no existe"
print(f"Probabilidades de test: {RUTA_PROBA_TEST} ({estado})")

Probabilidades de test: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\xgboost\probabilidades_test.csv (todavía no existe)


### Carga de test y verificaciones

66,544 filas, 1,175 ataques y las mismas 55 columnas que val, en el mismo orden.

In [6]:
X_test = pd.read_csv(RUTA_X_TEST)
y_test = pd.read_csv(RUTA_Y_TEST)["label_binaria"].to_numpy()

assert len(X_test) == N_TEST_ESPERADO, f"Se esperaban {N_TEST_ESPERADO:,} filas en X_test, hay {len(X_test):,}"
assert len(y_test) == N_TEST_ESPERADO, f"Se esperaban {N_TEST_ESPERADO:,} filas en y_test, hay {len(y_test):,}"
assert y_test.sum() == N_ATAQUES_TEST_ESPERADO, f"Se esperaban {N_ATAQUES_TEST_ESPERADO:,} ataques en test, hay {y_test.sum():,}"
assert list(X_test.columns) == COLUMNAS_VAL, "X_test no tiene las mismas 55 columnas, en el mismo orden, que X_val"

print(f"test: X {X_test.shape} | y {y_test.shape} | ataques: {y_test.sum()} ({y_test.mean():.4%})")

test: X (66544, 55) | y (66544,) | ataques: 1175 (1.7657%)


### Etiquetas por tipo de ataque, reconstruidas con el pipeline compartido

Igual que en `01_feature_engineering.ipynb`: `cargar_dataset_limpio` sobre `data/processed/wednesday_low_and_slow_clean.csv` y `train_test_split_by_flow` con los valores por defecto, ahora con `return_meta=True` para obtener la columna `Label` de cada fila. La función devuelve las tres particiones; acá solo se usa la de test.

Se verifica que la partición reconstruida es la misma que `X_test.csv` / `y_test.csv`, fila por fila: la etiqueta binaria tiene que coincidir con `y_test` y las 55 features con `X_test`. Además, test tiene que tener 915 Slowloris y 260 Slowhttptest, y una fila tiene etiqueta binaria 1 si y solo si su `Label` es uno de esos dos ataques.

También se guardan la etiqueta de índice de cada fila de test (`meta_test.index`) y su bloque. La etiqueta es la que usa la lista de filas comunes con la GRU. Como la partición reconstruida coincide fila por fila con `X_test.csv`, cada etiqueta queda en la misma posición que la probabilidad de esa fila en el archivo que se guarda después.

In [7]:
df_limpio = cargar_dataset_limpio(RUTA_DATASET_LIMPIO)
assert len(df_limpio) == N_DATASET_LIMPIO_ESPERADO, (
    f"Se esperaban {N_DATASET_LIMPIO_ESPERADO:,} filas en el dataset limpio, hay {len(df_limpio):,}"
)

_, _, X_test_rec, _, _, y_test_rec, _, _, meta_test = train_test_split_by_flow(df_limpio, return_meta=True)
del df_limpio

assert np.array_equal(y_test_rec.to_numpy(), y_test), "La etiqueta binaria reconstruida no coincide con y_test"
features_rec = X_test_rec[COLUMNAS_VAL].to_numpy(dtype=np.float64)
features_csv = X_test.to_numpy(dtype=np.float64)
assert np.allclose(features_rec, features_csv, rtol=1e-12, atol=0.0), "Las features reconstruidas no coinciden con X_test.csv"
print(f"\nPartición reconstruida = X_test.csv / y_test.csv, fila por fila "
      f"(features idénticas bit a bit: {np.array_equal(features_rec, features_csv)})")

tipo_test = meta_test[COLUMNA_ETIQUETA].to_numpy()
indice_test = meta_test.index.to_numpy()
bloque_test = X_test_rec["bloque"].to_numpy()
assert len(indice_test) == N_TEST_ESPERADO and len(np.unique(indice_test)) == N_TEST_ESPERADO, (
    "El índice de test reconstruido no tiene una etiqueta única por fila"
)
conteo_tipos = pd.Series(tipo_test).value_counts()
print(f"\nFlujos por etiqueta en test:\n{conteo_tipos.to_string()}")

assert set(conteo_tipos.index) == {ETIQUETA_BENIGNA, *ETIQUETAS_LOW_AND_SLOW}, (
    f"Etiquetas inesperadas en test: {sorted(conteo_tipos.index)}"
)
for tipo, n in N_POR_TIPO_ESPERADO.items():
    assert conteo_tipos[tipo] == n, f"Se esperaban {n} flujos {tipo} en test, hay {conteo_tipos[tipo]}"
assert np.array_equal(tipo_test != ETIQUETA_BENIGNA, y_test == 1), "Label y label_binaria no coinciden en test"
print("\nEtiquetas por tipo OK.")

del X_test_rec, y_test_rec, features_rec, features_csv

Fronteras de bloque (outlier_quantile=0.0, desempate=csv):
  Slowloris:    [2017-07-05 09:48:00 .. 2017-07-05 10:11:00]  (5790 filas)
  Slowhttptest: [2017-07-05 10:15:00 .. 2017-07-05 10:37:00]  (5499 filas)

Composición por bloque:
  Bloque 1: 53375 filas | Timestamp [2017-07-05 08:42:00 .. 2017-07-05 09:47:00] | ataques: 0 (0.00%)
  Bloque 2: 30362 filas | Timestamp [2017-07-05 09:48:00 .. 2017-07-05 10:11:00] | ataques: 5790 (19.07%)
  Bloque 3: 3578 filas | Timestamp [2017-07-05 10:12:00 .. 2017-07-05 10:14:00] | ataques: 0 (0.00%)
  Bloque 4: 33203 filas | Timestamp [2017-07-05 10:15:00 .. 2017-07-05 10:37:00] | ataques: 5499 (16.56%)
  Bloque 5: 323096 filas | Timestamp [2017-07-05 10:38:00 .. 2017-07-05 17:10:00] | ataques: 0 (0.00%)



Train: 310529 filas | Val: 66541 filas | Test: 66544 filas
Proporción de ataques -> train: 0.0265, val: 0.0283, test: 0.0177

Partición reconstruida = X_test.csv / y_test.csv, fila por fila (features idénticas bit a bit: True)

Flujos por etiqueta en test:
BENIGN              65369
DoS slowloris         915
DoS Slowhttptest      260

Etiquetas por tipo OK.


### Filas comunes con la GRU (verificación solo de datos, antes de predecir)

`idx_test_comun.npy` trae las etiquetas del índice (no posiciones) de las 66,454 filas de test que evaluó la GRU. Antes de predecir se verifica que el archivo:

- es un arreglo 1D de enteros, sin repetidos y con 66,454 etiquetas;
- tiene **todas** sus etiquetas en el índice de test reconstruido arriba.

Si alguna etiqueta no existe, el notebook se detiene: significaría que la lista se armó con otro split, o con posiciones en lugar de etiquetas.

Con la lista se arma `mascara_comun`, una máscara sobre las filas de test en el mismo orden que `X_test.csv` y que las probabilidades que se guardan después. Se reporta cuántas filas de test quedan fuera de la lista, su composición por tipo y su bloque. La máscara solo se usa en la sección 3, para filtrar las probabilidades guardadas; el modelo predice test completo una sola vez.

In [8]:
idx_comun = np.load(RUTA_IDX_COMUN, allow_pickle=False)
print(f"{RUTA_IDX_COMUN.relative_to(RAIZ_REPO)}: dtype {idx_comun.dtype}, shape {idx_comun.shape}")

assert idx_comun.ndim == 1, f"Se esperaba un arreglo 1D, tiene {idx_comun.ndim} dimensiones"
assert np.issubdtype(idx_comun.dtype, np.integer), f"Se esperaban etiquetas enteras, el dtype es {idx_comun.dtype}"
assert len(idx_comun) == N_COMUN_ESPERADO, f"Se esperaban {N_COMUN_ESPERADO:,} etiquetas, hay {len(idx_comun):,}"
assert len(np.unique(idx_comun)) == len(idx_comun), "La lista de filas comunes tiene etiquetas repetidas"
no_encontradas = np.setdiff1d(idx_comun, indice_test)
assert len(no_encontradas) == 0, (
    f"{len(no_encontradas):,} etiquetas de la lista no existen en el índice de test "
    f"(ejemplos: {no_encontradas[:10].tolist()}). Se detiene antes de predecir."
)

mascara_comun = np.isin(indice_test, idx_comun)
assert mascara_comun.sum() == N_COMUN_ESPERADO, "La máscara no selecciona exactamente las filas de la lista"
print(f"Las {N_COMUN_ESPERADO:,} etiquetas existen en el índice de test.")

ORDEN_TIPOS = [ETIQUETA_BENIGNA, *ETIQUETAS_LOW_AND_SLOW]
composicion = pd.DataFrame({
    "test completo": pd.Series(tipo_test).value_counts(),
    "en la lista (común)": pd.Series(tipo_test[mascara_comun]).value_counts(),
    "fuera de la lista": pd.Series(tipo_test[~mascara_comun]).value_counts(),
}).reindex(ORDEN_TIPOS).fillna(0).astype(int)
composicion.loc["total"] = composicion.sum()
print(f"\nFilas de test fuera de la lista: {int((~mascara_comun).sum()):,} de {len(indice_test):,}")
print(composicion.to_string())

fuera_por_bloque = pd.crosstab(pd.Series(bloque_test[~mascara_comun], name="bloque"),
                               pd.Series(tipo_test[~mascara_comun], name="tipo"))
print("\nFilas fuera de la lista, por bloque y tipo:")
print(fuera_por_bloque.to_string())

results\gru\03_evaluacion_test\idx_test_comun.npy: dtype int64, shape (66454,)
Las 66,454 etiquetas existen en el índice de test.



Filas de test fuera de la lista: 90 de 66,544
                  test completo  en la lista (común)  fuera de la lista
BENIGN                    65369                65290                 79
DoS slowloris               915                  914                  1
DoS Slowhttptest            260                  250                 10
total                     66544                66454                 90

Filas fuera de la lista, por bloque y tipo:
tipo    BENIGN  DoS Slowhttptest  DoS slowloris
bloque                                         
1           18                 0              0
2           17                 0              1
3           18                 0              0
4            8                10              0
5           18                 0              0


### Predicción sobre test (única vez, con guarda)

Esta es la única celda que aplica el modelo a test. Si `probabilidades_test.csv` ya existe, no predice. Si no existe, predice con `predict_xgboost_proba`, escribe las probabilidades junto con `label_binaria` en un archivo temporal, verifica que releído da exactamente las mismas probabilidades y recién entonces lo renombra a `probabilidades_test.csv`.

In [9]:
if RUTA_PROBA_TEST.exists():
    print(f"TEST YA FUE EVALUADO: existe {RUTA_PROBA_TEST}.\n"
          "No se vuelve a predecir. Las celdas siguientes leen las probabilidades de ese archivo.")
else:
    proba_predicha = predict_xgboost_proba(modelo, X_test)
    REPORTES_XGB_DIR.mkdir(parents=True, exist_ok=True)
    ruta_temporal = RUTA_PROBA_TEST.with_name(RUTA_PROBA_TEST.stem + "_tmp.csv")
    pd.DataFrame({"probabilidad": proba_predicha, "label_binaria": y_test}).to_csv(ruta_temporal, index=False)
    releidas = pd.read_csv(ruta_temporal, float_precision="round_trip")["probabilidad"].to_numpy()
    assert np.array_equal(releidas, proba_predicha), "Las probabilidades releídas no son idénticas a las predichas"
    ruta_temporal.replace(RUTA_PROBA_TEST)
    print(f"Predicción hecha y guardada en {RUTA_PROBA_TEST} ({len(proba_predicha):,} flujos).")
    del proba_predicha, releidas

Predicción hecha y guardada en C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\xgboost\probabilidades_test.csv (66,544 flujos).


### Probabilidades de test, leídas del archivo

Desde acá todo se calcula a partir del archivo guardado. Se verifica que tiene una fila por flujo de test y que su `label_binaria` coincide con `y_test`.

In [10]:
guardadas = pd.read_csv(RUTA_PROBA_TEST, float_precision="round_trip")
assert len(guardadas) == len(y_test), f"El archivo tiene {len(guardadas):,} filas, test tiene {len(y_test):,}"
assert np.array_equal(guardadas["label_binaria"].to_numpy(), y_test), "El archivo de probabilidades no corresponde a y_test"
proba_test = guardadas["probabilidad"].to_numpy(dtype=np.float64)
print(f"Probabilidades leídas de {RUTA_PROBA_TEST}: {len(proba_test):,} flujos")

Probabilidades leídas de C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\xgboost\probabilidades_test.csv: 66,544 flujos


## 3. Métricas en test

Todo con el modelo y el umbral ya cerrados. Nada de esta sección cambia el umbral, los hiperparámetros ni las features.

Las métricas salen de una sola función, `evaluar`, que se aplica dos veces con el mismo procedimiento: a test completo (3a) y al subconjunto común con la GRU (3b). El subconjunto sale de filtrar las probabilidades guardadas con `mascara_comun`. Para cada uno se calcula lo siguiente.

**Umbral principal (0.87).** `evaluate_model` escribe, tal cual lo produce, el JSON compartido para la comparación entre modelos:

- `reports/metrics/xgboost.json` para test completo;
- `reports/metrics/xgboost_comun.json` para el subconjunto común.

`save_dir` es la ruta absoluta de `reports/metrics/`, porque el valor por defecto es relativo y escribiría dentro de `notebooks/xgboost/`. `training_time_seconds=None` porque en este notebook el modelo no se entrena. `evaluate_model` incluye F2 (`f2_score`) pero no PR-AUC, que se calcula aparte con `average_precision_score`, igual que en `02_modelo.ipynb` y `03_tuning.ipynb`.

**Sensibilidad: umbrales 0.5 y 0.20 (no elegidos, solo sensibilidad).** El umbral del modelo es 0.87, fijado en val. 0.5 es el valor por defecto y 0.20 es la otra moda del bootstrap de `04_umbral.ipynb`. Solo muestran cuánto cambiarían las métricas con otro umbral; no se guardan en `reports/metrics/`.

**Recall por tipo de ataque**, con el umbral principal.

**Intervalos de confianza al 95 % por bootstrap.**

- 1,000 remuestreos con reemplazo, estratificados en tres estratos: benignos, Slowloris y Slowhttptest. Cada remuestreo conserva la cantidad de flujos de cada estrato, así que también conserva la cantidad de benignos y de ataques.
- En cada aplicación (test completo y subconjunto común) se crea un generador nuevo, `np.random.default_rng(42)`. En cada remuestreo se sortean los estratos en ese orden.
- El modelo no se vuelve a aplicar: se remuestrean las predicciones a umbral 0.87 ya calculadas.
- El intervalo es el de los percentiles 2.5 y 97.5 (`np.percentile`, interpolación lineal) de recall, precision, F1, F2, FP y del recall de cada tipo de ataque.

El intervalo mide la variabilidad debida a qué flujos tocaron en test, no la del entrenamiento.

In [11]:
N_BOOTSTRAP = 1000
SEMILLA = 42
ESTRATOS = [ETIQUETA_BENIGNA, *ETIQUETAS_LOW_AND_SLOW]
UMBRALES_SENSIBILIDAD = [0.5, 0.20]


def recall_por_tipo(y_pred: np.ndarray, tipos: np.ndarray) -> list[dict]:
    """Flujos, detectados, FN y recall de cada tipo de ataque."""
    filas = []
    for tipo in ETIQUETAS_LOW_AND_SLOW:
        es_tipo = tipos == tipo
        flujos = int(es_tipo.sum())
        detectados = int(y_pred[es_tipo].sum())
        filas.append({"tipo": tipo, "flujos": flujos, "detectados": detectados,
                      "FN": flujos - detectados, "recall": detectados / flujos})
    return filas


def ic95_bootstrap(y_pred: np.ndarray, tipos: np.ndarray, valores: dict) -> pd.DataFrame:
    """IC 95 % por bootstrap estratificado (benignos, Slowloris, Slowhttptest) de predicciones ya calculadas."""
    pred_por_estrato = {estrato: y_pred[tipos == estrato] for estrato in ESTRATOS}
    n_ataques = sum(len(pred_por_estrato[t]) for t in ETIQUETAS_LOW_AND_SLOW)

    rng = np.random.default_rng(SEMILLA)
    filas_boot = []
    for _ in range(N_BOOTSTRAP):
        positivos = {e: int(p[rng.integers(0, len(p), size=len(p))].sum()) for e, p in pred_por_estrato.items()}
        fp = positivos[ETIQUETA_BENIGNA]
        tp = sum(positivos[t] for t in ETIQUETAS_LOW_AND_SLOW)
        fn = n_ataques - tp
        fila = {
            "recall": tp / n_ataques,
            "precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
            "f1": 2 * tp / (2 * tp + fp + fn),
            "f2": 5 * tp / (5 * tp + 4 * fn + fp),
            "FP": fp,
        }
        for t in ETIQUETAS_LOW_AND_SLOW:
            fila[f"recall {t}"] = positivos[t] / len(pred_por_estrato[t])
        filas_boot.append(fila)
    bootstrap = pd.DataFrame(filas_boot)

    return pd.DataFrame({
        "valor": pd.Series(valores),
        "ic95_inf": bootstrap.apply(lambda c: np.percentile(c, 2.5)),
        "ic95_sup": bootstrap.apply(lambda c: np.percentile(c, 97.5)),
    })


def evaluar(y: np.ndarray, proba: np.ndarray, tipos: np.ndarray, nombre_metricas: str) -> dict:
    """Umbral principal (evaluate_model, que escribe el JSON compartido), PR-AUC, sensibilidad, recall por tipo
    e IC 95 % por bootstrap. Mismo procedimiento para test completo y para el subconjunto común."""
    assert np.array_equal(tipos != ETIQUETA_BENIGNA, y == 1), "Label y label_binaria no coinciden"
    y_pred = apply_threshold(proba)  # UMBRAL_FINAL por defecto
    resultados = evaluate_model(
        y,
        y_pred,
        y_proba=proba,
        model_name=nombre_metricas,
        training_time_seconds=None,
        save_dir=str(METRICAS_DIR),
    )
    mc = resultados["matriz_confusion"]
    principal = metricas_umbral(y, proba, UMBRAL_FINAL)
    assert (principal["TP"], principal["FP"], principal["FN"], principal["TN"]) == (
        mc["verdaderos_positivos"], mc["falsos_positivos"], mc["falsos_negativos"], mc["verdaderos_negativos"]
    ), "metricas_umbral y evaluate_model no dan la misma matriz de confusión"
    assert np.isclose(principal["f2"], resultados["f2_score"]), "F2 de metricas_umbral != f2_score de evaluate_model"

    filas_tipo = recall_por_tipo(y_pred, tipos)
    assert sum(f["detectados"] for f in filas_tipo) == mc["verdaderos_positivos"], "Los detectados por tipo no suman TP"
    valores = {
        "recall": float(resultados["recall"]),
        "precision": float(resultados["precision"]),
        "f1": float(resultados["f1_score"]),
        "f2": float(resultados["f2_score"]),
        "FP": mc["falsos_positivos"],
        **{f"recall {f['tipo']}": f["recall"] for f in filas_tipo},
    }
    return {
        "n": int(len(y)),
        "ataques": int(y.sum()),
        "resultados": resultados,
        "pr_auc": float(average_precision_score(y, proba)),
        "principal": principal,
        "sensibilidad": [metricas_umbral(y, proba, u) for u in UMBRALES_SENSIBILIDAD],
        "recall_por_tipo": filas_tipo,
        "ic95": ic95_bootstrap(y_pred, tipos, valores),
    }


def mostrar(ev: dict, titulo: str) -> None:
    mc = ev["resultados"]["matriz_confusion"]
    print(f"\nPR-AUC ({titulo}): {ev['pr_auc']:.4f}")

    matriz = pd.DataFrame(
        [[mc["verdaderos_negativos"], mc["falsos_positivos"]],
         [mc["falsos_negativos"], mc["verdaderos_positivos"]]],
        index=pd.Index(["benigno", "ataque"], name="real"),
        columns=pd.Index(["benigno", "ataque"], name="predicho"),
    )
    print(f"\nMatriz de confusión ({titulo}), umbral {UMBRAL_FINAL:g}:")
    print(matriz.to_string())

    tabla = pd.DataFrame([ev["principal"], *ev["sensibilidad"]])
    tabla.insert(1, "rol", ["principal (elegido en val)"] + ["no elegido, solo sensibilidad"] * len(ev["sensibilidad"]))
    print(f"\nUmbral principal y sensibilidad ({titulo}):")
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
        print(tabla[["umbral", "rol", "precision", "recall", "f1", "f2", "TP", "FP", "FN", "TN"]].to_string(index=False))

    print(f"\nRecall por tipo de ataque ({titulo}), umbral {UMBRAL_FINAL:g}:")
    with pd.option_context("display.float_format", "{:.4f}".format):
        print(pd.DataFrame(ev["recall_por_tipo"]).to_string(index=False))

    print(f"\nIC 95 % por bootstrap ({titulo}): {N_BOOTSTRAP} remuestreos, semilla {SEMILLA}, "
          f"estratos {', '.join(ESTRATOS)}")
    with pd.option_context("display.float_format", "{:.4f}".format):
        print(ev["ic95"].to_string())

### 3a. Test completo

In [12]:
ev_test = evaluar(y_test, proba_test, tipo_test, NOMBRE_METRICAS)
mostrar(ev_test, "test completo")

Métricas guardadas en: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\metrics\xgboost.json

=== Resultados: XGBoost ===
Accuracy:  0.9997
Precision: 0.9932
Recall:    0.9906  <- métrica principal
F1-score:  0.9919
F2-score:  0.9911
ROC-AUC:   0.9999
Matriz de confusión -> TN: 65361, FP: 8, FN: 11, TP: 1164



PR-AUC (test completo): 0.9977

Matriz de confusión (test completo), umbral 0.87:
predicho  benigno  ataque
real                     
benigno     65361       8
ataque         11    1164

Umbral principal y sensibilidad (test completo):
 umbral                           rol  precision  recall     f1     f2   TP  FP  FN    TN
 0.8700    principal (elegido en val)     0.9932  0.9906 0.9919 0.9911 1164   8  11 65361
 0.5000 no elegido, solo sensibilidad     0.9923  0.9923 0.9923 0.9923 1166   9   9 65360
 0.2000 no elegido, solo sensibilidad     0.9898  0.9932 0.9915 0.9925 1167  12   8 65357

Recall por tipo de ataque (test completo), umbral 0.87:
            tipo  flujos  detectados  FN  recall
   DoS slowloris     915         907   8  0.9913
DoS Slowhttptest     260         257   3  0.9885

IC 95 % por bootstrap (test completo): 1000 remuestreos, semilla 42, estratos BENIGN, DoS slowloris, DoS Slowhttptest
                         valor  ic95_inf  ic95_sup
recall                  0.990

### 3b. Subconjunto común con la GRU (66,454 filas)

Mismas métricas y mismo bootstrap que en 3a, sobre las filas de `idx_test_comun.npy`. Las probabilidades son las mismas que se guardaron para test completo, filtradas con `mascara_comun`; no hay una segunda predicción. El JSON compartido de este subconjunto es un archivo aparte, `reports/metrics/xgboost_comun.json`.

In [13]:
ev_comun = evaluar(y_test[mascara_comun], proba_test[mascara_comun], tipo_test[mascara_comun], NOMBRE_METRICAS_COMUN)
mostrar(ev_comun, "subconjunto común")


def fila_comparacion(ev: dict) -> dict:
    r, mc = ev["resultados"], ev["resultados"]["matriz_confusion"]
    return {
        "flujos": ev["n"], "ataques": ev["ataques"],
        "recall": r["recall"], "precision": r["precision"], "F1": r["f1_score"], "F2": r["f2_score"],
        "PR-AUC": ev["pr_auc"], "ROC-AUC": r["roc_auc"],
        "TP": mc["verdaderos_positivos"], "FP": mc["falsos_positivos"],
        "FN": mc["falsos_negativos"], "TN": mc["verdaderos_negativos"],
    }


comparacion = pd.DataFrame({"test completo": fila_comparacion(ev_test),
                            "subconjunto común": fila_comparacion(ev_comun)})
print(f"\nTest completo vs subconjunto común, umbral {UMBRAL_FINAL:g}:")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(comparacion.to_string())

Métricas guardadas en: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\metrics\xgboost_comun.json

=== Resultados: XGBoost_comun ===
Accuracy:  0.9997
Precision: 0.9931
Recall:    0.9905  <- métrica principal
F1-score:  0.9918
F2-score:  0.9911
ROC-AUC:   0.9999
Matriz de confusión -> TN: 65282, FP: 8, FN: 11, TP: 1153



PR-AUC (subconjunto común): 0.9977

Matriz de confusión (subconjunto común), umbral 0.87:
predicho  benigno  ataque
real                     
benigno     65282       8
ataque         11    1153

Umbral principal y sensibilidad (subconjunto común):
 umbral                           rol  precision  recall     f1     f2   TP  FP  FN    TN
 0.8700    principal (elegido en val)     0.9931  0.9905 0.9918 0.9911 1153   8  11 65282
 0.5000 no elegido, solo sensibilidad     0.9923  0.9923 0.9923 0.9923 1155   9   9 65281
 0.2000 no elegido, solo sensibilidad     0.9897  0.9931 0.9914 0.9924 1156  12   8 65278

Recall por tipo de ataque (subconjunto común), umbral 0.87:
            tipo  flujos  detectados  FN  recall
   DoS slowloris     914         906   8  0.9912
DoS Slowhttptest     250         247   3  0.9880

IC 95 % por bootstrap (subconjunto común): 1000 remuestreos, semilla 42, estratos BENIGN, DoS slowloris, DoS Slowhttptest
                         valor  ic95_inf  ic95_sup
recall   

### Resultados guardados

`reports/xgboost/resultados_test.json` guarda lo de esta sección para test completo y para el subconjunto común: métricas del umbral principal con PR-AUC, sensibilidad, recall por tipo e intervalos. Además guarda los datos del modelo (umbral, hiperparámetros, `scale_pos_weight` con sus conteos y best_iteration) y la composición de las filas que quedaron fuera de la lista. `reports/metrics/xgboost.json` y `reports/metrics/xgboost_comun.json` ya los escribió `evaluate_model`.

In [14]:
def a_json(ev: dict) -> dict:
    r = ev["resultados"]
    return {
        "n": ev["n"],
        "ataques": ev["ataques"],
        "flujos_por_tipo": {f["tipo"]: f["flujos"] for f in ev["recall_por_tipo"]},
        "metricas_umbral_principal": {
            "recall": float(r["recall"]),
            "precision": float(r["precision"]),
            "f1": float(r["f1_score"]),
            "f2": float(r["f2_score"]),
            "pr_auc": ev["pr_auc"],
            "roc_auc": float(r["roc_auc"]),
            "matriz_confusion": r["matriz_confusion"],
        },
        "sensibilidad_no_elegidos": ev["sensibilidad"],
        "recall_por_tipo": ev["recall_por_tipo"],
        "ic95_bootstrap": {
            "n_remuestreos": N_BOOTSTRAP,
            "semilla": SEMILLA,
            "estratos": ESTRATOS,
            "percentiles": [2.5, 97.5],
            "intervalos": {m: [float(ev["ic95"].loc[m, "ic95_inf"]), float(ev["ic95"].loc[m, "ic95_sup"])]
                           for m in ev["ic95"].index},
        },
    }


resultados_json = {
    "modelo": "XGBoost",
    "umbral": UMBRAL_FINAL,
    "hiperparametros": metadatos["hiperparametros"],
    "scale_pos_weight": metadatos["scale_pos_weight"],
    "n_benignos_train": metadatos["n_benignos_train"],
    "n_ataques_train": metadatos["n_ataques_train"],
    "best_iteration": metadatos["best_iteration"],
    "test_completo": a_json(ev_test),
    "subconjunto_comun_gru": {
        "archivo_indices": RUTA_IDX_COMUN.relative_to(RAIZ_REPO).as_posix(),
        "filas_fuera_de_la_lista": int((~mascara_comun).sum()),
        "composicion_fuera_de_la_lista": {t: int(composicion.loc[t, "fuera de la lista"]) for t in ORDEN_TIPOS},
        **a_json(ev_comun),
    },
    "tiempo_entrenamiento_segundos": None,
    "archivo_probabilidades": RUTA_PROBA_TEST.relative_to(RAIZ_REPO).as_posix(),
    "archivos_metricas_compartidas": [RUTA_METRICAS.relative_to(RAIZ_REPO).as_posix(),
                                      RUTA_METRICAS_COMUN.relative_to(RAIZ_REPO).as_posix()],
}
REPORTES_XGB_DIR.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS_TEST.write_text(json.dumps(resultados_json, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Resultados guardados en {RUTA_RESULTADOS_TEST}")
print(f"Métricas compartidas en {RUTA_METRICAS} y {RUTA_METRICAS_COMUN}")

Resultados guardados en C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\xgboost\resultados_test.json
Métricas compartidas en C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\metrics\xgboost.json y C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\reports\metrics\xgboost_comun.json


## 4. Resumen

In [15]:
def con_ic(ic95: pd.DataFrame, nombre: str, formato: str = ".4f") -> str:
    fila = ic95.loc[nombre]
    return f"{fila['valor']:{formato}} [{fila['ic95_inf']:{formato}}, {fila['ic95_sup']:{formato}}]"


def imprimir_bloque(ev: dict, titulo: str) -> None:
    mc, ic = ev["resultados"]["matriz_confusion"], ev["ic95"]
    flujos_por_tipo_txt = ", ".join(f"{f['tipo']} {f['flujos']}" for f in ev["recall_por_tipo"])
    print(f"\n{titulo}: {ev['n']:,} flujos | {ev['ataques']:,} ataques ({flujos_por_tipo_txt})")
    print(f"  Umbral principal {UMBRAL_FINAL:g} (IC 95 % bootstrap entre corchetes):")
    print(f"  recall     {con_ic(ic, 'recall')}")
    print(f"  precision  {con_ic(ic, 'precision')}")
    print(f"  F1         {con_ic(ic, 'f1')}")
    print(f"  F2         {con_ic(ic, 'f2')}")
    print(f"  FP         {con_ic(ic, 'FP', '.0f')} | FN {mc['falsos_negativos']} | TP {mc['verdaderos_positivos']} | "
          f"TN {mc['verdaderos_negativos']}")
    print(f"  PR-AUC {ev['pr_auc']:.4f} | ROC-AUC {ev['resultados']['roc_auc']:.4f}")
    for f in ev["recall_por_tipo"]:
        print(f"  recall {f['tipo']:<17} {con_ic(ic, 'recall ' + f['tipo'])} ({f['detectados']}/{f['flujos']})")
    print("  Sensibilidad (no elegidos, solo sensibilidad):")
    for m in ev["sensibilidad"]:
        print(f"    umbral {m['umbral']:<5g} recall {m['recall']:.4f} | precision {m['precision']:.4f} | "
              f"F1 {m['f1']:.4f} | F2 {m['f2']:.4f} | FP {m['FP']} | FN {m['FN']}")


print("=" * 72)
print("RESUMEN — Fase 7: evaluación final XGBoost en test")
print("=" * 72)
print(f"Modelo:          congelado en models/xgboost, no reentrenado | lr {HIPERPARAMETROS['learning_rate']} | "
      f"max_depth {HIPERPARAMETROS['max_depth']} | min_child_weight {HIPERPARAMETROS['min_child_weight']} | "
      f"scale_pos_weight {metadatos['scale_pos_weight']:.4f} | best_iteration {modelo.best_iteration}")
print(f"Sanidad en val:  FP {obtenido_val['FP']}, FN {obtenido_val['FN']} a umbral {UMBRAL_FINAL:g}")
imprimir_bloque(ev_test, "Test completo")
imprimir_bloque(ev_comun, f"Subconjunto común con la GRU ({RUTA_IDX_COMUN.name})")
fuera_txt = ", ".join(f"{t} {int(composicion.loc[t, 'fuera de la lista'])}" for t in ORDEN_TIPOS)
print(f"  Filas de test fuera de la lista: {int((~mascara_comun).sum()):,} ({fuera_txt})")
print(f"\nArchivos: {RUTA_METRICAS}\n          {RUTA_METRICAS_COMUN}\n          {RUTA_RESULTADOS_TEST}\n"
      f"          {RUTA_PROBA_TEST}")

RESUMEN — Fase 7: evaluación final XGBoost en test
Modelo:          congelado en models/xgboost, no reentrenado | lr 0.3 | max_depth 4 | min_child_weight 1 | scale_pos_weight 36.7313 | best_iteration 68
Sanidad en val:  FP 11, FN 3 a umbral 0.87

Test completo: 66,544 flujos | 1,175 ataques (DoS slowloris 915, DoS Slowhttptest 260)
  Umbral principal 0.87 (IC 95 % bootstrap entre corchetes):
  recall     0.9906 [0.9847, 0.9966]
  precision  0.9932 [0.9881, 0.9974]
  F1         0.9919 [0.9880, 0.9953]
  F2         0.9911 [0.9862, 0.9957]
  FP         8 [3, 14] | FN 11 | TP 1164 | TN 65361
  PR-AUC 0.9977 | ROC-AUC 0.9999
  recall DoS slowloris     0.9913 [0.9847, 0.9967] (907/915)
  recall DoS Slowhttptest  0.9885 [0.9731, 1.0000] (257/260)
  Sensibilidad (no elegidos, solo sensibilidad):
    umbral 0.5   recall 0.9923 | precision 0.9923 | F1 0.9923 | F2 0.9923 | FP 9 | FN 9
    umbral 0.2   recall 0.9932 | precision 0.9898 | F1 0.9915 | F2 0.9925 | FP 12 | FN 8

Subconjunto común con l